In [17]:
# импорты библиотек
import gspread
import pandas as pd
from datetime import datetime, date
from google.oauth2.service_account import Credentials
import re
from dotenv import load_dotenv
load_dotenv()

# Константы подключения
SPREADSHEET_ID = "1wxNIhrPjoFQf3vqatu9oK78W5RmNoI3YG6sf9i4GWAQ"

# Основные листы + листы "Уволенные" и "Стаффинг" (для полноты данных)
SHEET_NAMES = [
    "DEMO Лист", "A-Benefits", "1C", "Active Bank", "A-Sber", 
    "Betta Bank", "Evergreen Bank+Laba ABS", "Liberty Bank", 
    "Investment", "L &D", "MyCare", "SQL Trainee", "ProBusiness",
    "Уволенные 2026", "Стаффинг"
]

# Отдельный лист для промежуточных статусов
INTERIM_STATUS_SHEET = "Промежуточный статус стажера"

# Индексы столбцов для основных листов (начинаются с 0)
COL_TECH = 1      # B
COL_TECHLEAD = 2  # C
COL_NAME = 3      # D
COL_LEVEL = 4     # E (Уровень экспертности / Экспертность)
COL_START = 5     # F
COL_STOP = 6      # G
COL_REASON = 7    # H

# Индексы столбцов для листа "Промежуточный статус стажера"
INTERIM_COL_TECH = 2         # C
INTERIM_COL_TECHLEAD = 3     # D
INTERIM_COL_NAME = 4         # E
INTERIM_COL_START = 6        # G
INTERIM_COL_STOP = 7         # H
INTERIM_COL_REASON = 8       # I

# ==========================================
# СПРАВОЧНИКИ ДЛЯ РАСЧЕТА ЗП (из Excel)
# ==========================================
# Примечание: clean_tech_name приводит названия к верхнему регистру и убирает пробелы/спецсимволы.
# Поэтому 'BA/SA' -> 'BASA', 'QA Engenier' -> 'QAENGENIER'
RATES = {
    'BASA': 25000, 'BA/SA': 25000,
    'JAVA': 22000,
    'QAJAVA' : 23000,
    'REACT': 17250,
    'DEVOPS': 25000,
    'QAENGENIER': 23000, 'QAENGINEER': 23000, 'AQAJAVA': 23000,
    'ARCHITECTOR': 27000, 'АРХИТЕКТОР': 27000
}

COEFFICIENTS = {
    'L1': 0.0,
    'L2': 1.0,
    'L3': 1.5
}

# Магический делитель из формулы (выявлен обратным инжинирингом)
# Рекомендуется вынести в конфиг, если он может меняться
DIVISOR = 88.0 

# Настройки отчетного периода
REPORT_YEAR = 2026
REPORT_MONTH = 10  # Октябрь

In [18]:
def authenticate_with_service_account(credentials_file: str) -> gspread.Client:
    scopes = [
        "https://www.googleapis.com/auth/spreadsheets.readonly",
        "https://www.googleapis.com/auth/drive.readonly"
    ]
    creds = Credentials.from_service_account_file(credentials_file, scopes=scopes)
    return gspread.authorize(creds)

def clean_date(val) -> date | None:
    """Преобразует значение в объект даты"""
    if pd.isna(val) or str(val).strip() == '':
        return None
    if isinstance(val, datetime):
        return val.date()
    if isinstance(val, date):
        return val
    
    val_str = str(val).strip()
    for fmt in ['%d.%m.%Y', '%Y-%m-%d', '%d/%m/%Y', '%m/%d/%Y', '%Y-%m-%dT%H:%M:%S']:
        try:
            return datetime.strptime(val_str, fmt).date()
        except ValueError:
            continue
    return None

def parse_date_range(val) -> tuple[date | None, date | None]:
    """Парсит дату или диапазон дат"""
    if pd.isna(val) or str(val).strip() == '':
        return None, None
    
    val_str = str(val).strip()
    if '-' in val_str and '.' in val_str:
        parts = val_str.split('-')
        if len(parts) == 2:
            start_date = clean_date(parts[0].strip())
            end_date = clean_date(parts[1].strip())
            return start_date, end_date
    return clean_date(val_str), None

def clean_fio(val) -> str:
    """Очищает ФИО от лишних символов"""
    if pd.isna(val) or str(val).strip() == '':
        return ''
    val_str = str(val).strip()
    match = re.match(r'^[A-Za-zА-Яа-яЁё\s\-]+', val_str)
    if match:
        return match.group(0).strip()
    return val_str

def normalize_fio(fio: str) -> str:
    """Нормализует ФИО для корректного подсчета уникальных людей"""
    if not fio:
        return ''
    return re.sub(r'\s+', ' ', fio.lower().strip()).title()

def clean_tech_name(tech_name):
    """Нормализует название технологии"""
    if pd.isna(tech_name):
        return tech_name
    
    tech_str = str(tech_name).upper()
    cyrillic_to_latin = str.maketrans({
        'А': 'A', 'В': 'B', 'Е': 'E', 'К': 'K', 'М': 'M',
        'Н': 'H', 'О': 'O', 'Р': 'P', 'С': 'C', 'Т': 'T', 'Х': 'X',
    })
    tech_str = tech_str.translate(cyrillic_to_latin)
    tech_str = re.sub(r'[^A-Z0-9]', '', tech_str)
    tech_str = re.sub(r'^Q?AQ?AJAVA$', 'AQAJAVA', tech_str)
    return tech_str

def is_valid_techlead(name: str) -> bool:
    """Проверяет, что имя техлида выглядит как реальное 'Имя Фамилия'"""
    if pd.isna(name):
        return False
    name = str(name).strip()
    if len(name) < 3 or " " not in name:
        return False
    blacklist = ["-", "х", "x", "2-я технология", "вторая технология", "команда", "techleads"]
    if name.lower() in blacklist:
        return False
    return True

def extract_sheet_data(worksheet) -> pd.DataFrame:
    """Парсит основные листы гугл дока"""
    values = worksheet.get_values()
    if not values:
        return pd.DataFrame()
    
    header_idx = -1
    for i, row in enumerate(values):
        if len(row) > COL_NAME:
            col_b = str(row[COL_TECH]).strip().lower()
            col_c = str(row[COL_TECHLEAD]).strip().lower()
            col_d = str(row[COL_NAME]).strip().lower()
            
            is_tech = 'технология' in col_b
            is_techlead = 'техлид' in col_c or 'techlead' in col_c
            is_name = 'фамилия' in col_d or 'фио' in col_d
            
            if is_tech and is_techlead and is_name:
                header_idx = i
                break
    
    if header_idx == -1:
        return pd.DataFrame()

    data = []
    for i in range(header_idx + 1, len(values)):
        row = values[i]
        while len(row) <= COL_REASON:
            row.append('')
        
        col_b_val = str(row[COL_TECH]).strip()
        if not col_b_val:
            continue
            
        fio = clean_fio(row[COL_NAME]).strip()
        if not fio:
            continue

        # Извлекаем Уровень (колонка E), если его нет - по умолчанию L2
        level_val = str(row[COL_LEVEL]).strip().upper() if len(row) > COL_LEVEL else 'L2'
        if level_val not in COEFFICIENTS:
            level_val = 'L2' # Фоллбэк на случай мусора в ячейке

        data.append({
            'Технология': col_b_val,
            'Техлид': clean_fio(row[COL_TECHLEAD]),
            'Уровень': level_val,
            'Лаборант': fio,
            'Дата старта': clean_date(row[COL_START]),
            'Дата стопа': clean_date(row[COL_STOP]),
            'Причина_статус': str(row[COL_REASON]).strip().lower(),
            'Источник': worksheet.title
        })
    
    return pd.DataFrame(data)

def extract_interim_status_data(worksheet) -> pd.DataFrame:
    """Парсит лист 'Промежуточный статус стажера'"""
    values = worksheet.get_values()
    if not values:
        return pd.DataFrame()
    
    header_idx = -1
    for i, row in enumerate(values):
        if len(row) > INTERIM_COL_NAME:
            col_c = str(row[INTERIM_COL_TECH]).strip().lower()
            col_d = str(row[INTERIM_COL_TECHLEAD]).strip().lower()
            col_e = str(row[INTERIM_COL_NAME]).strip().lower()
            
            is_tech = 'технология' in col_c
            is_techlead = 'техлид' in col_d or 'techlead' in col_d
            is_name = 'фамилия' in col_e or 'фио' in col_e
            
            if is_tech and is_techlead and is_name:
                header_idx = i
                break
    
    if header_idx == -1:
        return pd.DataFrame()

    data = []
    for i in range(header_idx + 1, len(values)):
        row = values[i]
        while len(row) <= INTERIM_COL_REASON:
            row.append('')
        
        col_c_val = str(row[INTERIM_COL_TECH]).strip()
        if not col_c_val:
            continue
            
        fio = clean_fio(row[INTERIM_COL_NAME]).strip()
        if not fio:
            continue

        reason = str(row[INTERIM_COL_REASON]).strip().lower()
        
        if 'неоплачиваемый отпуск' in reason:
            start_date, end_date = parse_date_range(row[INTERIM_COL_STOP])
            
            data.append({
                'Технология': col_c_val,
                'Техлид': clean_fio(row[INTERIM_COL_TECHLEAD]),
                'Уровень': 'L2', # Для отпусков уровень не критичен, т.к. они исключаются
                'Лаборант': fio,
                'Дата старта': start_date if start_date else clean_date(row[INTERIM_COL_START]),
                'Дата стопа': end_date,
                'Причина_статус': reason,
                'Источник': worksheet.title
            })
    
    return pd.DataFrame(data)

In [19]:
def generate_payroll_reports(raw_df, interim_df, year, month):
    df = raw_df.copy()
    
    # 1. Нормализация технологий и уровней
    df['Технология'] = df['Технология'].apply(clean_tech_name)
    df['Техлид'] = df['Техлид'].apply(clean_fio)
    df = df[df['Техлид'].apply(is_valid_techlead)].copy()
    df['Лаборант_норм'] = df['Лаборант'].apply(normalize_fio)
    
    # 2. Конвертация дат
    df['Дата старта'] = pd.to_datetime(df['Дата старта'], errors='coerce')
    df['Дата стопа'] = pd.to_datetime(df['Дата стопа'], errors='coerce')
    df = df.dropna(subset=['Дата старта'])
    
    # 3. Обработка неоплачиваемых отпусков
    unpaid_leave_periods = []
    if interim_df is not None and not interim_df.empty:
        interim_df['Дата старта'] = pd.to_datetime(interim_df['Дата старта'], errors='coerce')
        interim_df['Дата стопа'] = pd.to_datetime(interim_df['Дата стопа'], errors='coerce')
        interim_df['Лаборант_норм'] = interim_df['Лаборант'].apply(normalize_fio)
        interim_df['Технология'] = interim_df['Технология'].apply(clean_tech_name)
        interim_df['Техлид'] = interim_df['Техлид'].apply(clean_fio)
        
        for _, row in interim_df.iterrows():
            if pd.notna(row['Дата старта']) and pd.notna(row['Дата стопа']):
                unpaid_leave_periods.append({
                    'Лаборант_норм': row['Лаборант_норм'],
                    'Технология': row['Технология'],
                    'Техлид': row['Техлид'],
                    'start': row['Дата старта'],
                    'end': row['Дата стопа']
                })
    
    # 4. Генерация ежедневных записей
    start_date = pd.Timestamp(year=year, month=month, day=1)
    end_date = start_date + pd.DateOffset(months=1) - pd.DateOffset(days=1)
    dates_range = pd.date_range(start_date, end_date)
    
    daily_records = []
    
    for current_date in dates_range:
        mask = (df['Дата старта'] <= current_date) & \
               ((df['Дата стопа'] >= current_date) | df['Дата стопа'].isna())
        
        active_df = df[mask].copy()
        
        if unpaid_leave_periods:
            for period in unpaid_leave_periods:
                if period['start'] <= current_date <= period['end']:
                    exclude_mask = (
                        (active_df['Лаборант_норм'] == period['Лаборант_норм']) &
                        (active_df['Технология'] == period['Технология']) &
                        (active_df['Техлид'] == period['Техлид'])
                    )
                    active_df = active_df[~exclude_mask]
        
        if not active_df.empty:
            # Группируем с учетом Уровня экспертности
            counts = active_df.groupby(['Технология', 'Техлид', 'Уровень'])['Лаборант_норм'].nunique().reset_index(name='количество лаборантов')
            counts['дата'] = current_date.date()
            daily_records.append(counts)
            
    if daily_records:
        daily_df = pd.concat(daily_records, ignore_index=True)
        daily_df = daily_df[['Технология', 'Техлид', 'Уровень', 'дата', 'количество лаборантов']]
        daily_df = daily_df.sort_values(by=['дата', 'Технология', 'Техлид']).reset_index(drop=True)
    else:
        daily_df = pd.DataFrame(columns=['Технология', 'Техлид', 'Уровень', 'дата', 'количество лаборантов'])
        
    # ==========================================
    # РАСЧЕТ ЗП (Логика из Excel)
    # ==========================================
    periods_data = []
    
    # Ищем непрерывные периоды (когда кол-во стажеров и уровень не меняются)
    for (tech, lead, level), group in daily_df.groupby(['Технология', 'Техлид', 'Уровень']):
        group = group.sort_values('дата').reset_index(drop=True)
        
        current_start = group.iloc[0]['дата']
        current_end = group.iloc[0]['дата']
        current_count = group.iloc[0]['количество лаборантов']
        
        for i in range(1, len(group)):
            row = group.iloc[i]
            next_date = pd.to_datetime(row['дата'])
            next_count = row['количество лаборантов']
            
            # Проверяем, идет ли день за днем и не менялось ли кол-во людей
            is_consecutive = (next_date - pd.to_datetime(current_end)).days == 1
            is_same_count = (next_count == current_count)
            
            if is_consecutive and is_same_count:
                current_end = next_date
            else:
                periods_data.append({
                    'Технология': tech, 'Техлид': lead, 'Уровень': level,
                    'Начало периода': pd.to_datetime(current_start).date(),
                    'Конец периода': pd.to_datetime(current_end).date(),
                    'Количество стажеров': current_count,
                    'Дней в периоде': (pd.to_datetime(current_end) - pd.to_datetime(current_start)).days + 1
                })
                current_start = next_date
                current_end = next_date
                current_count = next_count
                
        # Добавляем последний период
        periods_data.append({
            'Технология': tech, 'Техлид': lead, 'Уровень': level,
            'Начало периода': pd.to_datetime(current_start).date(),
            'Конец периода': pd.to_datetime(current_end).date(),
            'Количество стажеров': current_count,
            'Дней в периоде': (pd.to_datetime(current_end) - pd.to_datetime(current_start)).days + 1
        })
        
    periods_df = pd.DataFrame(periods_data)
    
    if not periods_df.empty:
        # Маппинг рейтов и коэффициентов
        periods_df['Рейт'] = periods_df['Технология'].map(RATES).fillna(0)
        periods_df['Коэффициент'] = periods_df['Уровень'].map(COEFFICIENTS).fillna(1.0)
        
        # Формула: (Рейт * Коэффициент * Дней * Стажеров) / 88
        periods_df['Итоговая сумма за период'] = (
            (periods_df['Рейт'] * periods_df['Коэффициент'] * periods_df['Дней в периоде'] * periods_df['Количество стажеров']) / DIVISOR
        ).round(2)
        
        # Итоговая сводка ЗП по Технологиям и Техлидам
        payroll_summary = periods_df.groupby(['Технология', 'Техлид', 'Уровень']).agg(
            Итоговая_сумма_за_месяц=('Итоговая сумма за период', 'sum')
        ).reset_index()
    else:
        periods_df = pd.DataFrame(columns=['Технология', 'Техлид', 'Уровень', 'Начало периода', 'Конец периода', 'Количество стажеров', 'Дней в периоде', 'Рейт', 'Коэффициент', 'Итоговая сумма за период'])
        payroll_summary = pd.DataFrame(columns=['Технология', 'Техлид', 'Уровень', 'Итоговая_сумма_за_месяц'])

    # ==========================================
    # Сводка по дням (старая логика)
    # ==========================================
    summary_df = daily_df.groupby(['Технология', 'Техлид', 'Уровень']).agg(
        дней_с_лаборантами=('дата', 'count'),
        всего_лаборанто_дней=('количество лаборантов', 'sum')
    ).reset_index()
    
    period_days = len(dates_range)
    summary_df['среднее_лаборантов_в_день'] = (
            summary_df['всего_лаборанто_дней'] / max(period_days, 1)
    ).round(2)
    
    summary_df = summary_df.sort_values(by=['Технология', 'Техлид']).reset_index(drop=True)
    
    return daily_df, summary_df, periods_df, payroll_summary

In [20]:
def main():
    print("Аутентификация через Service Account...")
    try:
        client = authenticate_with_service_account('credentials.json')
        print("Аутентификация успешна")
    except Exception as e:
        print(f"Ошибка аутентификации: {e}")
        return

    print("Открытие таблицы...")
    try:
        spreadsheet = client.open_by_key(SPREADSHEET_ID)
        print(f"Таблица открыта: {spreadsheet.title}")
    except Exception as e:
        print(f"ОШИБКА открытия таблицы: {e}")
        return

    all_data = []
    for sheet_name in SHEET_NAMES:
        try:
            worksheet = spreadsheet.worksheet(sheet_name)
            df = extract_sheet_data(worksheet)
            if not df.empty:
                all_data.append(df)
                print(f"  Лист '{sheet_name}': {len(df)} строк")
        except Exception as e:
            print(f"  Ошибка на листе {sheet_name}: {e}")

    interim_df = pd.DataFrame()
    try:
        worksheet = spreadsheet.worksheet(INTERIM_STATUS_SHEET)
        interim_df = extract_interim_status_data(worksheet)
        if not interim_df.empty:
            print(f"  Лист '{INTERIM_STATUS_SHEET}': {len(interim_df)} записей о неоплачиваемых отпусках")
    except Exception as e:
        print(f"  Ошибка на листе {INTERIM_STATUS_SHEET}: {e}")

    if not all_data:
        print("ОШИБКА: Нет данных для сохранения.")
        return

    raw_df = pd.concat(all_data, ignore_index=True)
    print(f"\nУспешно спарсено {len(raw_df)} строк до фильтрации.")

    # Генерация отчетов (теперь возвращает 4 датафрейма)
    daily_df, summary_df, periods_df, payroll_summary = generate_payroll_reports(raw_df, interim_df, REPORT_YEAR, REPORT_MONTH)
    
    filename = f'payroll_report_{REPORT_YEAR}_{REPORT_MONTH:02d}.xlsx'
    
    with pd.ExcelWriter(filename, engine='openpyxl', datetime_format='YYYY-MM-DD') as writer:
        # 1. Ежедневный отчет
        daily_df.to_excel(writer, sheet_name='Ежедневный_отчет', index=False)
        
        # 2. Сводка по дням
        summary_df.to_excel(writer, sheet_name='Сводка_за_месяц', index=False)
        
        # 3. ДЕТАЛИЗАЦИЯ ЗП (Периоды, как в вашем Excel-примере)
        periods_df.to_excel(writer, sheet_name='Детализация_ЗП', index=False)
        
        # 4. ИТОГИ ЗП (Общие суммы к выплате)
        payroll_summary.to_excel(writer, sheet_name='Итоги_ЗП', index=False)
        
        # Автоширина колонок для всех листов
        for sheet_name in writer.sheets:
            worksheet = writer.sheets[sheet_name]
            for i, col in enumerate(worksheet.columns):
                max_len = max(len(str(cell.value)) for cell in col if cell.value is not None)
                worksheet.column_dimensions[chr(65 + i)].width = min(max_len + 2, 35)

    print(f"\nОтчет успешно сохранен в файл: {filename}")
    print(f" - Лист 'Ежедневный_отчет': {len(daily_df)} строк")
    print(f" - Лист 'Сводка_за_месяц': {len(summary_df)} строк")
    print(f" - Лист 'Детализация_ЗП': {len(periods_df)} периодов")
    print(f" - Лист 'Итоги_ЗП': {len(payroll_summary)} итоговых сумм")

if __name__ == "__main__":
    main()

Аутентификация через Service Account...
Аутентификация успешна
Открытие таблицы...
Таблица открыта: Копия ФИДБЕКИ Aston
  Лист 'DEMO Лист': 16 строк
  Лист 'A-Benefits': 16 строк
  Лист '1C': 43 строк
  Лист 'Active Bank': 51 строк
  Лист 'A-Sber': 30 строк
  Лист 'Betta Bank': 30 строк
  Лист 'Evergreen Bank+Laba ABS': 52 строк
  Лист 'Liberty Bank': 57 строк
  Лист 'Investment': 46 строк
  Ошибка на листе L &D: L &D
  Лист 'MyCare': 27 строк
  Лист 'SQL Trainee': 22 строк
  Лист 'ProBusiness': 21 строк
  Лист 'Промежуточный статус стажера': 1 записей о неоплачиваемых отпусках

Успешно спарсено 411 строк до фильтрации.

Отчет успешно сохранен в файл: payroll_report_2026_10.xlsx
 - Лист 'Ежедневный_отчет': 2883 строк
 - Лист 'Сводка_за_месяц': 93 строк
 - Лист 'Детализация_ЗП': 121 периодов
 - Лист 'Итоги_ЗП': 93 итоговых сумм
